# 03. SQL Safety & Security Controls

Covers **Attribute 9**:
- Read-only SQLite connection
- Single-statement SELECT-only validation
- Table whitelisting
- Blocked SQL keywords (DROP, DELETE, UPDATE, INSERT, ALTER, ATTACH, PRAGMA)
- Row cap enforcement (LIMIT 500)

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if "high_level" in str(pathlib.Path.cwd()) or "capabilities" in str(pathlib.Path.cwd()) else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.orchestrator import Orchestrator
from src.llm_client import MockLLMClient
from src.tools.sql_tool import run_query, validate_sql
from src.tools.retrieval_tool import get_index
from src.tools.code_tool import run_code
from src.formatting import format_value, rows_to_markdown_table

print("AB InBev Enterprise Q&A Agent Pipeline Loaded.")

AB InBev Enterprise Q&A Agent Pipeline Loaded.

### 1. Valid Read-Only SELECT Execution

In [2]:
res = run_query("SELECT brand, SUM(net_revenue_usd) AS total_rev FROM fact_monthly_kpi GROUP BY brand LIMIT 5")
print(f"Rows returned: {res.row_count}")
print(f"Columns: {res.columns}")
for row in res.rows:
    print(f"  {row[0]}: ${row[1]:,.0f}")

Rows returned: 5
Columns: ['brand', 'total_rev']
  Brahma: $53,066,919
  Bud Light: $48,475,579
  Budweiser: $49,923,845
  Corona: $145,180,792
  Corona Cero: $51,215,360

### 2. Adversarial Attacks Safely Blocked

In [3]:
from src.tools.sql_tool import SQLSafetyError

attacks = [
    "SELECT * FROM fact_monthly_kpi; DROP TABLE dim_brand;",
    "DELETE FROM fact_monthly_kpi WHERE 1=1",
    "UPDATE fact_monthly_kpi SET net_revenue_usd = 0",
    "SELECT * FROM sqlite_master",
    "PRAGMA table_info(dim_brand)"
]
for attack in attacks:
    try:
        validate_sql(attack)
        print(f"FAIL: Attack allowed: {attack}")
    except SQLSafetyError as e:
        print(f"BLOCKED: {attack}\n  Reason: {e}\n")

BLOCKED: SELECT * FROM fact_monthly_kpi; DROP TABLE dim_brand;
  Reason: Multiple statements are not allowed.

BLOCKED: DELETE FROM fact_monthly_kpi WHERE 1=1
  Reason: Only SELECT (or WITH ... SELECT) statements are allowed.

BLOCKED: UPDATE fact_monthly_kpi SET net_revenue_usd = 0
  Reason: Only SELECT (or WITH ... SELECT) statements are allowed.

BLOCKED: SELECT * FROM sqlite_master
  Reason: Query references unknown/disallowed table(s): ['sqlite_master']

BLOCKED: PRAGMA table_info(dim_brand)
  Reason: Only SELECT (or WITH ... SELECT) statements are allowed.


### 3. Automatic Row Cap Enforcement

In [4]:
queries = [
    "SELECT brand FROM fact_monthly_kpi",
    "SELECT brand FROM fact_monthly_kpi LIMIT 999999",
    "SELECT brand FROM fact_monthly_kpi LIMIT 10"
]
for q in queries:
    safe = validate_sql(q)
    print(f"Original: {q}\nSanitized: {safe}\n")

Original: SELECT brand FROM fact_monthly_kpi
Sanitized: SELECT brand FROM fact_monthly_kpi LIMIT 500

Original: SELECT brand FROM fact_monthly_kpi LIMIT 999999
Sanitized: SELECT brand FROM fact_monthly_kpi LIMIT 500

Original: SELECT brand FROM fact_monthly_kpi LIMIT 10
Sanitized: SELECT brand FROM fact_monthly_kpi LIMIT 10
